# 12 · Orchestrator-workers and evaluator-optimizer loops
Level L3 · Part 3 · professional · ~15 min · builds on 11

## Goal
Split one customer message across workers with an orchestrator. Then add an evaluator that checks the draft
against written criteria, sends it back once with the problems named, and stops on a rule when the draft still fails.

## Why it matters
One customer message often holds two jobs: "Where is order 1093?" needs an order lookup, and "Can I get my money back
for order 2207?" needs the returns policy. One prompt that does both hides which part went wrong, and nothing checks
the reply before the customer reads it.

| Without a split and a check | With an orchestrator and an evaluator |
|---|---|
| One prompt does every job; a wrong part hides | Each subtask has its own answer you can test |
| The first draft goes straight to the customer | An evaluator checks the draft against criteria first |
| "Your refund is approved" goes out unchecked | The evaluator names the problem; a person approves refunds |

**What it makes easy:** one place to see which job failed, and one fixed rule for a bad draft. **When you don't need
it:** a question with one answer and no check, or a fixed workflow whose steps you know in advance (lesson 11).

## The idea
In an **orchestrator-workers** flow, a central model reads the message, decides the subtasks, and hands each one to a
worker. A worker can be plain code (an order lookup) or a model with its own context. The results are then combined
into one draft. In an **evaluator-optimizer** loop, one call writes a draft and a second call judges it against
criteria. If the judge fails it, its problems go back to the writer. A **stop rule** (a maximum number of rounds)
ends the loop, and a draft that never passes goes to a person, never to the customer.

Analogy: a newsroom. An editor splits a story into research tasks, reporters work in parallel, a writer drafts, and a
copy editor checks the draft against the style sheet before it is published.

## Picture
```mermaid
flowchart LR
  M[Customer message] --> O[Orchestrator: plans the subtasks, call 1]
  O --> W1[Worker order_status: code]
  O --> W2[Worker returns_policy: code]
  W1 --> D[Optimizer: writes the draft, call 2]
  W2 --> D
  D --> J{Evaluator: judges the draft, call 3}
  J -->|pass| S[Send the reply]
  J -->|fail, with problems| R[Optimizer revises, call 4]
  R --> J2{Evaluator judges again, call 5}
  J2 -->|pass| S
  J2 -->|fail| P[Stop rule: a person writes the reply]
```

## Step by step
**Step 1 · setup and data (one client for the lesson).** The key is read from `.env`, never typed here. The model is
called through the official `google-genai` SDK: `client.aio` is its async interface, so calls can wait at once.
`llm_replay` is a course helper, not part of the SDK: it saves each model call the first time, so later runs replay the
same answers. `tut_env.env` is a course helper that reads `.env`. The model id is pinned, as in lessons 02 to 11.

In [1]:
import sys; sys.path.insert(0, "..")
import asyncio
import llm_replay  # course helper: records and replays model calls (not part of the SDK)
llm_replay.start("12_orchestrator-and-evaluator")

from typing import Literal
from google import genai  # the official Google SDK
from google.genai import types
from pydantic import BaseModel
from tut_env import env  # course helper: reads a setting from the .env file (not part of the SDK)

MODEL = "gemini-3.1-flash-lite"  # the course's pinned model, as in lessons 02 to 11
client = genai.Client(  # in your own project keep vertexai=True and pass your own key
    vertexai=True,
    api_key="replay" if llm_replay.is_replaying() else env("GOOGLE_CLOUD_API_KEY"),
    http_options=types.HttpOptions(timeout=30_000, retry_options=types.HttpRetryOptions(
        attempts=4, initial_delay=5.0, max_delay=30.0)),  # the SDK's own retry for 429 and 5xx
)
CALLS = []  # every model call this lesson makes, in order: the 5-call budget and the bill
# True = replaying the saved recording (no key needed); False only while a new recording is made with a key.
print("client ready; replaying a recording:", llm_replay.is_replaying())

LLM mode: replay (12_orchestrator-and-evaluator.yaml)


client ready; replaying a recording: True


**Step 1b · the data.** The policy is a short text written for this course; it agrees with the 30-day rule. The
orders are the fictional rows of lessons 06 and 11. The customer's message holds two jobs on purpose.

In [2]:
POLICY = """Brightside returns policy (written for this course; it agrees with the 30-day rule).
Items can be returned within 30 days of delivery, unused and in their original packaging.
Refunds are approved by a person on the support team. The support desk never promises a refund.
Damaged items: the customer sends a photo, and a person decides on a replacement or a refund."""
ORDERS = {  # the same fictional rows as lessons 06 and 11
    "1093": {"status": "in transit", "days_late": 3},
    "2207": {"status": "delivered", "days_late": 0},
}
MESSAGE = ("Where is order 1093? It should have arrived three days ago. "
           "Also, if I return the jacket from order 2207, can I get my money back?")
print(POLICY, "\n\ncustomer:", MESSAGE)

Brightside returns policy (written for this course; it agrees with the 30-day rule).
Items can be returned within 30 days of delivery, unused and in their original packaging.
Refunds are approved by a person on the support team. The support desk never promises a refund.
Damaged items: the customer sends a photo, and a person decides on a replacement or a refund. 

customer: Where is order 1093? It should have arrived three days ago. Also, if I return the jacket from order 2207, can I get my money back?


**Step 2 · the orchestrator plans (call 1).** The orchestrator's answer is a schema, `Plan`, so the code can check it.
The worker names are fixed in the schema: the model can pick a worker, but not invent one. The code also caps the
number of subtasks, because each subtask may cost a call. If a plan is longer than the cap, the code raises an error
and the message goes to a person, not to the workers.

In [3]:
class Subtask(BaseModel):
    worker: Literal["order_status", "returns_policy"]  # the only workers that exist
    question: str
    order_id: str | None = None  # used by order_status only


class Plan(BaseModel):
    subtasks: list[Subtask]


MAX_SUBTASKS = 3  # the code refuses a longer plan: a cap on cost, as in lesson 06


async def orchestrate(message: str) -> Plan:
    prompt = ("You are the orchestrator of the Brightside support desk. Split the customer message into subtasks, "
              "one question each. Use worker 'order_status' for one order (give its order_id) and 'returns_policy' "
              "for a return, refund or delivery question. Do not answer the questions yourself.\n\n"
              f"Customer message: {message}")
    CALLS.append("plan")
    resp = await client.aio.models.generate_content(  # call 1
        model=MODEL, contents=prompt,
        config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json",
                                           response_json_schema=Plan.model_json_schema()))
    plan = Plan.model_validate_json(resp.text)
    if len(plan.subtasks) > MAX_SUBTASKS:
        raise ValueError(f"the plan has {len(plan.subtasks)} subtasks; the cap is {MAX_SUBTASKS}")
    return plan


plan = await orchestrate(MESSAGE)
for s in plan.subtasks:
    print(s.worker, "| order", s.order_id, "|", s.question)

order_status | order 1093 | Where is order 1093? It should have arrived three days ago.
returns_policy | order 2207 | If I return the jacket from order 2207, can I get my money back?


**Step 3 · the workers run at the same time.** Both workers here are plain code: `order_status` reads the order rows,
and `returns_policy` returns the policy text (it is four lines, so the worker returns all of it; lesson 17 retrieves only
the matching part). A worker can also be a model with its own context, and then it costs a call. This lesson spends its
five-call budget on the loop instead. `asyncio.gather` starts the workers together. With `return_exceptions=True`, one
failed worker comes back as an error object and does not hide the others, so the code converts those errors to values
before any draft uses them.

In [4]:
def lookup_order(order_id: str) -> dict:
    """Code worker: a plain function, no model call."""
    return ORDERS.get(order_id, {"error": f"no order with id {order_id}"})


async def run_worker(sub: Subtask):
    if sub.worker == "order_status":
        return lookup_order(sub.order_id)
    return {"policy": POLICY}  # code worker: the whole short policy, no model call


facts = await asyncio.gather(*(run_worker(s) for s in plan.subtasks), return_exceptions=True)
# A worker that raised is an exception object here: turn it into a value the drafter can read, not a crash.
facts = [f if not isinstance(f, Exception) else {"error": str(f)} for f in facts]
for s, f in zip(plan.subtasks, facts):
    print(s.worker, "->", f)

order_status -> {'status': 'in transit', 'days_late': 3}
returns_policy -> {'policy': 'Brightside returns policy (written for this course; it agrees with the 30-day rule).\nItems can be returned within 30 days of delivery, unused and in their original packaging.\nRefunds are approved by a person on the support team. The support desk never promises a refund.\nDamaged items: the customer sends a photo, and a person decides on a replacement or a refund.'}


**Step 4 · the optimizer writes, the evaluator judges (calls 2 and 3).** The writer and the judge are two calls with two
jobs. The judge does not write the reply, and it checks the draft against criteria written down as a list. Its
answer is a schema too, so `pass` or `fail` is a value the code can branch on.

In [5]:
class Verdict(BaseModel):
    verdict: Literal["pass", "fail"]
    problems: list[str]  # empty when the draft passes


CRITERIA = ("1. Answers every question in the customer message. 2. Uses only the facts given; adds none. "
            "3. Promises no refund: a person approves refunds. 4. Four sentences at most.")


async def write_draft(facts, feedback: str = "") -> str:
    """The optimizer: writes a reply, and fixes the named problems when a judge sent feedback."""
    prompt = ("Write the reply to the customer. Be warm and reassuring.\n"
              f"Customer message: {MESSAGE}\nFacts from the workers: {facts}\n")
    if feedback:
        prompt += f"A reviewer found these problems in the last draft. Fix them: {feedback}\n"
    CALLS.append("draft")
    resp = await client.aio.models.generate_content(  # call 2 or 4
        model=MODEL, contents=prompt, config=types.GenerateContentConfig(temperature=0))
    return resp.text


async def judge(draft: str, facts) -> Verdict:
    """The evaluator: checks one draft against the criteria. It never writes the reply."""
    prompt = (f"Check the draft against these criteria: {CRITERIA}\nFacts: {facts}\nDraft: {draft}\n"
              "Give verdict pass or fail, and list the problems (empty when it passes).")
    CALLS.append("judge")
    resp = await client.aio.models.generate_content(  # call 3 or 5
        model=MODEL, contents=prompt,
        config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json",
                                           response_json_schema=Verdict.model_json_schema()))
    return Verdict.model_validate_json(resp.text)

**Step 5 · the loop and its stop rule (calls 2 to 5).** The loop draws, judges, and on a fail sends the problems back to
the writer. `MAX_ROUNDS = 2` is the stop rule: at most two drafts, each judged. If neither passes, the code sends no
reply and a person writes it. The decision for each case is the one you write in Try it. The budget is a maximum of 5;
this recording used all 5. A run where round 1 passes uses 3 calls. The count also includes retries: the SDK's
`HttpRetryOptions(attempts=4)` retries a busy-server error (429) or a 5xx on its own, and each retry is another request
on the bill.

In [6]:
MAX_ROUNDS = 2  # the stop rule: at most two drafts, each judged; then a person


async def optimize(facts):
    feedback = ""
    for n in range(1, MAX_ROUNDS + 1):
        draft = await write_draft(facts, feedback)  # the optimizer
        verdict = await judge(draft, facts)         # the evaluator
        print(f"round {n} draft: {draft}\nround {n} verdict: {verdict.verdict} {verdict.problems}")
        if verdict.verdict == "pass":
            return draft, f"round {n}: passed"
        feedback = "; ".join(verdict.problems)
    return None, f"{MAX_ROUNDS} rounds, none passed: a person writes the reply"


reply, status = await optimize(facts)
print("status:", status)
print("reply:", reply if reply else "(none: the stop rule handed it to a person)")
print(f"model calls: {len(CALLS)} -> {CALLS}")
llm_replay.stop()  # the last model call of this notebook

round 1 draft: Subject: Update regarding your orders 1093 and 2207

Hi [Customer Name],

Thank you for reaching out to us. I am so sorry to hear that your order #1093 hasn’t arrived yet. I completely understand how frustrating it is to wait for a package, especially when it’s already a few days past the expected delivery date.

I have checked the status of your shipment, and I can confirm that it is currently in transit. While it is running about three days behind schedule, it is on its way to you. I will keep a close eye on the tracking information and send you an update as soon as I see any movement.

Regarding your question about the jacket from order #2207, I would be happy to help you with that. Under our Brightside returns policy, you are welcome to return items within 30 days of delivery, provided they are unused and in their original packaging. 

Once you send the item back, our support team will review the return to process your refund request. Please just let me know if you’d

## At work
- Anthropic's essay "Building effective agents" (2024-12-19) describes orchestrator-workers for "complex tasks where you
  can't predict the subtasks needed", and evaluator-optimizer when there are "clear evaluation criteria". The plan
  is model output, so teams cap and log it → lesson 33 (cost caps).
- A judge is a model too, and it can pass a bad draft. Teams measure the judge against labelled examples before they trust
  it → lesson 22 (LLM-as-judge), and run the checks on every change → lesson 28 (evals in CI).
- A fan-out waits for its slowest worker, and each model worker can time out. Give every branch a timeout and a retry
  → lesson 32 (reliability). Lesson 19 draws the same loop as a graph with state.

## Common mistakes
1. **Asking the drafter to grade its own draft.** A writer that checks itself in the same prompt tends to pass its own
   work. Use a separate call with separate criteria, as in Step 4.
2. **Forgetting `await` on an async judge.** The call returns a coroutine, not a result. The cell below raises on
   purpose, with no model call.

In [7]:
async def check_draft(draft: str) -> dict:
    return {"verdict": "fail" if "refund" in draft.lower() else "pass"}


verdict = check_draft("Your refund is approved.")  # no await: verdict is a coroutine
print(verdict["verdict"])  # raises: a coroutine is not a dict

TypeError: 'coroutine' object is not subscriptable

The fix is `await check_draft(...)`. A notebook runs top-level `await`, as the cells above do.

3. **A loop with no stop rule.** `while verdict != "pass":` can spend money for ever when the writer and the judge
   disagree. Always cap the rounds, and decide what happens when the cap is reached.

## Try it
Write `next_step(verdict, round_no, max_rounds)`. It returns `"send"` when the draft passed, `"revise"` when it failed
and rounds are left, and `"person"` when it failed on the last round. No model call is needed.

In [8]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [9]:
def next_step(verdict: str, round_no: int, max_rounds: int) -> str:
    if verdict == "pass":
        return "send"
    return "revise" if round_no < max_rounds else "person"


print(next_step("pass", 1, 2))  # send
print(next_step("fail", 1, 2))  # revise
print(next_step("fail", 2, 2))  # person

send
revise
person


## Recap
- An orchestrator reads the message, plans the subtasks and hands each one to a worker. Here both workers were code; a
  model worker would cost one call each.
- Workers that do not depend on each other run at the same time with `asyncio.gather`; `return_exceptions=True` keeps one
  failure from hiding the rest.
- An evaluator checks a draft against written criteria. It is a separate call from the writer.
- A loop needs a stop rule: a maximum number of rounds and a safe end when the cap is reached.

**Interview:** "When would you add an evaluator to a support agent?" Strong answer: "When there are clear criteria the
draft can fail, such as a promise the desk may not make. The judge is a separate call with a cap on rounds, and an
unpassed draft goes to a person. I measure the judge against labelled examples before I trust it."